# Problem Framing: Empirical Hormone Reference Ranges

Hormone reference ranges used in clinical practice are typically derived from reference populations
that are poorly defined, inconsistently filtered, and rarely stratified by age in a way that
reflects known physiological variation. The result is a set of thresholds — printed on lab reports
and used to guide treatment decisions — that may misclassify healthy individuals as abnormal, or
mask genuine pathology behind an overly broad "normal" band.

The core problem was articulated clearly by Travison et al. (2017), who showed that harmonized
testosterone reference ranges derived from carefully screened healthy men differed substantially
from those reported by major commercial laboratories. Their exclusion criteria — removing subjects
with obesity (BMI ≥ 30), diabetes, opioid use, hypertension, and reproductive disorders — produced
ranges that were narrower, higher, and age-dependent. The Endocrine Society's 2018 clinical practice
guideline (Bhasin et al.) subsequently recommended that reference ranges be established from
well-characterized healthy populations using mass spectrometry assays, rather than from convenience
samples processed with immunoassay platforms of variable accuracy.

This notebook frames the problem, reviews the key literature, and demonstrates the empirical
methodology on synthetic data calibrated to published values. The goal is to validate the
computational approach before applying it to real NHANES data in Phase 2 of this project.

## Reference literature

| Citation | Contribution | Why it matters here |
|---|---|---|
| Travison TG, et al. (2017). Harmonized reference ranges for circulating testosterone levels in men of four cohort studies in the United States and Europe. *JCEM*, 102(4): 1161–1173. | Established age-stratified testosterone reference ranges from a pooled healthy male cohort (N=9,054) using LC-MS/MS, with explicit exclusion criteria for obesity, diabetes, and other confounders. | Provides the exclusion criteria and published percentile values that this project replicates and extends using NHANES. |
| Bhasin S, et al. (2018). Testosterone Therapy in Men With Hypogonadism: An Endocrine Society Clinical Practice Guideline. *JCEM*, 103(5): 1715–1744. | Clinical practice guideline recommending diagnosis of hypogonadism using testosterone measured by LC-MS/MS, with reference ranges derived from healthy young men. | Sets the clinical standard this project's empirical ranges will be compared against. Motivates the need for population-level, age-stratified norms. |
| Vesper HW, et al. (2014). Interlaboratory comparison study of serum total testosterone measurements performed by mass spectrometry methods. *Steroids*, 79: 35–44. | Quantified inter-laboratory variability in testosterone measurement even among mass spectrometry methods, showing CVs of 5–15% depending on concentration. | Establishes that assay variability is a real confounder. NHANES uses a single standardized LC-MS/MS protocol, which partly controls for this. |
| Mitchell M, et al. (2019). Model Cards for Model Reporting. *Proceedings of FAT* 2019*. | Proposed a structured reporting format for ML models that includes intended use, limitations, ethical considerations, and performance metrics by subgroup. | The model card format will be used for every versioned reference range artifact this project produces. Limitations sections are a feature, not a disclaimer. |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Clean plot style: white background, no grid, readable fonts
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
        "figure.figsize": (8, 5),
        "figure.dpi": 120,
    }
)

## Synthetic data calibrated to published values

Before applying this methodology to real NHANES data, we demonstrate it on synthetic data whose
parameters are drawn from published values. This serves two purposes: it validates that the
computational pipeline produces sensible outputs, and it provides a reference implementation
that reviewers can run without needing access to the NHANES download.

The synthetic population below is calibrated to Travison et al. (2017), Table 2. Total testosterone
in healthy men follows an approximately log-normal distribution, with a mean that declines roughly
1% per year after age 30 and within-age variance that increases modestly with age. We generate
N=2,000 synthetic male subjects aged 20–80 and sample testosterone values from this parameterization.
The resulting distributions are not real data and should not be cited as such — they are a
methodological scaffold.

In [ ]:
np.random.seed(42)

n = 2000
ages = np.random.randint(20, 81, size=n)

# Log-normal parameters calibrated to Travison 2017 Table 2
# Baseline ~600 ng/dL at age 30, declining ~1% per year after 30
baseline_mean = 600  # ng/dL at age 30
decline_rate = 0.01  # 1% per year after age 30

age_adjusted_mean = np.where(
    ages <= 30,
    baseline_mean,
    baseline_mean * (1 - decline_rate) ** (ages - 30),
)

# Log-normal: if X ~ LogNormal(mu, sigma), then E[X] = exp(mu + sigma^2/2)
# We want E[X] = age_adjusted_mean with reasonable CV ~25%
cv = 0.25
sigma = np.sqrt(np.log(1 + cv**2))
mu = np.log(age_adjusted_mean) - sigma**2 / 2

testosterone = np.random.lognormal(mean=mu, sigma=sigma, size=n)

df = pd.DataFrame(
    {
        "subject_id": np.arange(1, n + 1),
        "age": ages,
        "sex": "M",
        "testosterone_ng_dl": np.round(testosterone, 1),
    }
)

print(df.head(10))
print()
print(df.describe())

In [ ]:
from scipy.stats.mstats import mquantiles

# Bin ages into decades
df["age_decade"] = (df["age"] // 10) * 10
df["age_label"] = df["age_decade"].apply(lambda d: f"{d}–{d+9}")

percentiles = (
    df.groupby("age_label")["testosterone_ng_dl"]
    .apply(
        lambda x: pd.Series(
            mquantiles(x, prob=[0.025, 0.5, 0.975]),
            index=["p2.5", "median", "p97.5"],
        )
    )
    .unstack()
    .round(1)
)

# Add sample size
percentiles["n"] = df.groupby("age_label")["testosterone_ng_dl"].count()
percentiles = percentiles[["n", "p2.5", "median", "p97.5"]]

print(percentiles)

In [ ]:
fig, ax = plt.subplots()

decades = percentiles.index
x = range(len(decades))

ax.plot(x, percentiles["median"], color="#2563eb", linewidth=2, label="Median")
ax.plot(
    x,
    percentiles["p2.5"],
    color="#2563eb",
    linewidth=1,
    linestyle="--",
    label="2.5th percentile",
)
ax.plot(
    x,
    percentiles["p97.5"],
    color="#2563eb",
    linewidth=1,
    linestyle="--",
    label="97.5th percentile",
)
ax.fill_between(
    x,
    percentiles["p2.5"],
    percentiles["p97.5"],
    color="#2563eb",
    alpha=0.12,
)

ax.set_xticks(x)
ax.set_xticklabels(decades)
ax.set_xlabel("Age decade")
ax.set_ylabel("Testosterone (ng/dL)")
ax.set_title("Empirical testosterone ranges from synthetic data, by age decade")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

decades = percentiles.index
x = range(len(decades))

# Empirical ranges
ax.plot(
    x,
    percentiles["median"],
    color="#2563eb",
    linewidth=2,
    label="Empirical median",
)
ax.plot(
    x,
    percentiles["p2.5"],
    color="#2563eb",
    linewidth=1,
    linestyle="--",
    label="Empirical 2.5th / 97.5th",
)
ax.plot(x, percentiles["p97.5"], color="#2563eb", linewidth=1, linestyle="--")
ax.fill_between(
    x,
    percentiles["p2.5"],
    percentiles["p97.5"],
    color="#2563eb",
    alpha=0.12,
)

# LabCorp adult male reference range: 264-916 ng/dL (flat across all ages)
ax.axhline(
    y=264,
    color="#e8544a",
    linewidth=1.5,
    linestyle="--",
    label="LabCorp lower (264 ng/dL)",
)
ax.axhline(
    y=916,
    color="#e8544a",
    linewidth=1.5,
    linestyle="--",
    label="LabCorp upper (916 ng/dL)",
)

# Annotate divergence
ax.annotate(
    "Empirical lower bound exceeds\nclinical lower bound in younger decades",
    xy=(1, percentiles["p2.5"].iloc[1]),
    xytext=(2.5, 200),
    fontsize=9,
    arrowprops=dict(arrowstyle="->", color="gray"),
    color="gray",
)

ax.set_xticks(x)
ax.set_xticklabels(decades)
ax.set_xlabel("Age decade")
ax.set_ylabel("Testosterone (ng/dL)")
ax.set_title("Empirical (synthetic) vs. published clinical ranges")
ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

## Implications and next steps

Even with synthetic data calibrated to published age-stratified values, the empirical ranges differ
substantially from the flat clinical ranges used by major commercial laboratories. The 2.5th
percentile in younger decades sits well above LabCorp's 264 ng/dL lower bound, suggesting that
a 25-year-old man with a testosterone of 300 ng/dL would be classified as "normal" by the lab
report but would fall below the 2.5th percentile of a healthy age-matched population. Conversely,
the empirical upper bound in older decades falls below the 916 ng/dL clinical ceiling, meaning the
flat range overstates what is physiologically typical for a 70-year-old. These are exactly the
patterns Travison et al. (2017) documented in real data.

Phase 2 of this project applies the same methodology to real NHANES data (cycles H and I), with
explicit healthy-subsample exclusion criteria following Travison 2017: BMI < 30, no self-reported
diabetes, no opioid use, no hypertension medication, no reproductive disorders. The plots above
are a methodological preview, not a result claim. The empirical percentiles, bootstrap confidence
intervals, and model card that will accompany the real analysis are the actual deliverables. This
notebook exists to show that the computational approach is sound before we commit to conclusions
drawn from it.

## References

1. Travison TG, Vesper HW, Orwoll E, et al. Harmonized reference ranges for circulating testosterone levels in men of four cohort studies in the United States and Europe. *Journal of Clinical Endocrinology & Metabolism*. 2017;102(4):1161–1173. doi:10.1210/jc.2016-2935

2. Bhasin S, Brito JP, Cunningham GR, et al. Testosterone Therapy in Men With Hypogonadism: An Endocrine Society Clinical Practice Guideline. *Journal of Clinical Endocrinology & Metabolism*. 2018;103(5):1715–1744. doi:10.1210/jc.2018-00229

3. Vesper HW, Botelho JC, Shacklady C, Smith A, Myers GL. Interlaboratory comparison study of serum total testosterone measurements performed by mass spectrometry methods. *Steroids*. 2014;82:35–44. doi:10.1016/j.steroids.2014.01.005

4. Mitchell M, Wu S, Zaldivar A, et al. Model Cards for Model Reporting. *Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT*)*. 2019:220–229. doi:10.1145/3287560.3287596